In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

### 读取数据集并查看

In [ ]:
train_data = pd.read_csv(r"DataSet\train.csv")
test_data = pd.read_csv(r"DataSet\test.csv")
train_data

In [ ]:
# 下面的代码是为了填充数据中的缺失值
for col in train_data.columns:
    if col == "SalePrice" or col == "Id":   # 跳过标签列和ID列
        continue
    if pd.api.types.is_numeric_dtype(train_data[col]):  # 判断是否为连续数值型数据
        fill_value = train_data[col].mean()
        # 保持整数类型
        if pd.api.types.is_integer_dtype(train_data[col]):
            fill_value = int(round(fill_value))
        else:
            # 保留原有小数位数
            decimals = train_data[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
            fill_value = round(float(fill_value), decimals)
        train_data[col].fillna(fill_value, inplace=True)
    else:  # 如果是分类数据
        fill_value = train_data[col].mode()[0]
        train_data[col].fillna(fill_value, inplace=True)

# 对测试数据也进行相同的处理
for col in test_data.columns:
    if col == "Id":   # 跳过ID列
        continue
    if pd.api.types.is_numeric_dtype(test_data[col]):  # 判断是否为连续数值型数据
        fill_value = test_data[col].mean()
        if pd.api.types.is_integer_dtype(test_data[col]):
            fill_value = int(round(fill_value))
        else:
            decimals = test_data[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
            fill_value = round(float(fill_value), decimals)
        test_data[col].fillna(fill_value, inplace=True)
    else:  # 如果是分类数据
        fill_value = test_data[col].mode()[0]
        test_data[col].fillna(fill_value, inplace=True)

In [ ]:
noise_example = train_data.sample(1).copy()
train_mean = {col: train_data[col].mean() for col in train_data.columns if pd.api.types.is_numeric_dtype(train_data[col])}
train_std = {col: train_data[col].std() for col in train_data.columns if pd.api.types.is_numeric_dtype(train_data[col])}
#用字典存储对应的事先计算的均值与标准差，避免反复计算

for col in noise_example.columns:
    if col == "Id":
        noise_example[col] = train_data[col].max() + 1
    elif pd.api.types.is_numeric_dtype(train_data[col]):
        value = np.random.normal(loc=train_mean[col], scale=train_std[col])
        # 保证生成数据非负，因为一间房子的面积之类的数值不能为负数是吧
        value = np.clip(value, 0, None)
        # 保持格式
        if pd.api.types.is_integer_dtype(train_data[col]):
            value = int(round(value))
        else:
            # 保留与原数据相同的小数位数
            decimals = train_data[col].apply(lambda x: str(x)[::-1].find('.') if '.' in str(x) else 0).max()
            value = round(float(value), decimals)
        noise_example[col] = value
    else:
        noise_example[col] = np.random.choice(train_data[col])

noise_example
